# BTC Price Action ResearchThis notebook is structured as a readable Price Action research flow for BTC aligned with `SNR_Line.pine`.- load fresh market data using `auto / binance / yfinance_now`- inspect market structure, liquidity sweeps, breakouts, CHoCH, FVG and SNR reactions- explain what each Price Action pattern means- generate potential signal candidates like `liquidity sweep` and `breakout`- use AI / model blocks only as a secondary confirmation layer- export bridge payload for TradingView

## 1. ImportsThe notebook uses both research tools and the same bridge utilities that feed the indicator.

In [ ]:
from datetime import datetime, timezonefrom pathlib import Pathimport jsonimport timeimport warningsimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as snsfrom IPython.display import clear_output, displayfrom sklearn.ensemble import HistGradientBoostingClassifierfrom sklearn.inspection import permutation_importancefrom sklearn.metrics import roc_auc_score, average_precision_score, classification_reportfrom sklearn.metrics import RocCurveDisplay, PrecisionRecallDisplay, ConfusionMatrixDisplayfrom snr_ml_features import FeatureConfig, load_market_data, add_snr_features, build_impulse_targetsfrom snr_ml_features import default_feature_columns, make_training_framefrom integration_bridge import build_signal_payload, payload_to_indicator_fields, build_pine_bridge_payloadfrom binance_live_stream import BinanceKlineStreamfrom screenshot_reader import choose_latest_screenshot, analyze_screenshotwarnings.filterwarnings("ignore")sns.set_theme(style="darkgrid")plt.rcParams["figure.figsize"] = (16, 6)pd.set_option("display.max_columns", 120)

## 2. Config And Fresh DataThis section gives three practical modes:- `auto`: Binance first, then yfinance fallback- `yfinance_now`: fetch using `datetime.now(timezone.utc)` and explicit `start/end`- `CSV_PATH`: use local history when you need longer backtests

In [ ]:
DATA_DIR = Path.cwd()OUT_DIR = DATA_DIR / "outputs"OUT_DIR.mkdir(parents=True, exist_ok=True)CSV_PATH = None  # Example: DATA_DIR / "btc_15m.csv"SYMBOL = "BTC-USD"INTERVAL = "15m"PERIOD = "45d"TIMEZONE = "Asia/Seoul"DATA_SOURCE = "auto"  # auto | binance | yfinance | yfinance_nowYF_USE_DATETIME_NOW = TrueLIVE_REFRESH_SECONDS = 30cfg = FeatureConfig(    symbol=SYMBOL,    interval=INTERVAL,    period=PERIOD,    timezone=TIMEZONE,    data_source=DATA_SOURCE,    snr_length=200,    outer_band_width=20,    target_horizon=12,    target_threshold=700.0,)def load_research_frame(data_source=None, csv_path=CSV_PATH, use_yf_now=YF_USE_DATETIME_NOW):    source = data_source or cfg.data_source    end_now_utc = datetime.now(timezone.utc)    raw_df = load_market_data(        csv_path=str(csv_path) if csv_path else None,        symbol=cfg.symbol,        interval=cfg.interval,        period=cfg.period,        timezone=cfg.timezone,        data_source=source,        end_datetime=end_now_utc,        use_yfinance_now=use_yf_now,    )    feature_df = add_snr_features(raw_df, config=cfg)    feature_df = build_impulse_targets(feature_df, horizon=cfg.target_horizon, move_threshold=cfg.target_threshold)    return feature_df, end_now_utcdef load_yfinance_now_frame():    return load_research_frame(data_source="yfinance_now", use_yf_now=True)df, pulled_at_utc = load_research_frame()summary = pd.Series({    "data_source": cfg.data_source,    "pulled_at_utc": pulled_at_utc.isoformat(),    "rows": len(df),    "start": str(df.index.min()),    "end": str(df.index.max()),    "last_close": round(float(df["close"].dropna().iloc[-1]), 2),    "large_move_rate_pct": round(df["large_move"].mean() * 100, 2),    "ny_move_rate_pct": round(df.loc[df["is_ny"] == 1, "large_move"].mean() * 100, 2),    "silver_move_rate_pct": round(df.loc[df["is_silver"] == 1, "large_move"].mean() * 100, 2),})display(summary)

## 3. Yfinance Now SnapshotThis separate snapshot explicitly uses `datetime.now(timezone.utc)` with yfinance, which is useful when you want the most recent Yahoo-compatible pull even if the main source is `auto`.

In [ ]:
yf_now_df, yf_now_utc = load_yfinance_now_frame()display(pd.Series({    "yf_now_pulled_at_utc": yf_now_utc.isoformat(),    "rows": len(yf_now_df),    "start": str(yf_now_df.index.min()),    "end": str(yf_now_df.index.max()),    "last_close": round(float(yf_now_df["close"].dropna().iloc[-1]), 2),}))

## 4. Current Market SnapshotThis cell turns the latest row into an easy-to-read state summary before the deeper charts.

In [ ]:
last = df.dropna().iloc[-1]state_snapshot = pd.Series({    "timestamp": str(last.name),    "close": round(float(last["close"]), 2),    "reg_basis": round(float(last["reg_basis"]), 2),    "snr_upper": round(float(last["snr_upper"]), 2),    "snr_lower": round(float(last["snr_lower"]), 2),    "poc_proxy": round(float(last["poc_proxy"]), 2),    "rsi": round(float(last["rsi"]), 2),    "flow_main": round(float(last["flow_main"]), 2),    "in_discount": int(last["in_discount"]),    "in_premium": int(last["in_premium"]),    "is_asia": int(last["is_asia"]),    "is_london": int(last["is_london"]),    "is_ny": int(last["is_ny"]),    "is_london_kz": int(last["is_london_kz"]),    "is_ny_kz": int(last["is_ny_kz"]),    "is_silver": int(last["is_silver"]),    "bull_break": int(last["bull_break"]),    "bear_break": int(last["bear_break"]),    "bull_choch": int(last["bull_choch"]),    "bear_choch": int(last["bear_choch"]),    "in_bull_fvg": int(last["in_bull_fvg"]),    "in_bear_fvg": int(last["in_bear_fvg"]),    "snr_long_reclaim": int(last["snr_long_reclaim"]),    "snr_short_reject": int(last["snr_short_reject"]),    "kz_long_score": round(float(last["kz_long_score"]), 2),    "kz_short_score": round(float(last["kz_short_score"]), 2),})display(state_snapshot)

## 5. Price Action ReferenceBefore looking at charts, this block explains the main PA signals used in the notebook.

In [ ]:
pa_reference = pd.DataFrame([    {"pattern": "Bull Sweep", "meaning": "Liquidity taken below prior lows and price reclaims back up", "potential_signal": "Potential long if followed by CHoCH, FVG or SNR reclaim"},    {"pattern": "Bear Sweep", "meaning": "Liquidity taken above prior highs and price rejects back down", "potential_signal": "Potential short if followed by CHoCH, FVG or SNR reject"},    {"pattern": "Bull Breakout", "meaning": "Price expands above structure or SNR upper boundary", "potential_signal": "Potential continuation long, best when backed by trend and flow"},    {"pattern": "Bear Breakout", "meaning": "Price expands below structure or SNR lower boundary", "potential_signal": "Potential continuation short, best when backed by trend and flow"},    {"pattern": "Bull CHoCH", "meaning": "Structure shifts from bearish to bullish", "potential_signal": "Potential reversal long if sweep / discount / FVG agree"},    {"pattern": "Bear CHoCH", "meaning": "Structure shifts from bullish to bearish", "potential_signal": "Potential reversal short if sweep / premium / FVG agree"},    {"pattern": "SNR Long Reclaim", "meaning": "Price sweeps or touches support area and closes back above basis", "potential_signal": "Potential long reclaim entry"},    {"pattern": "SNR Short Reject", "meaning": "Price touches resistance area and closes back below basis", "potential_signal": "Potential short reject entry"},    {"pattern": "Bull FVG Active", "meaning": "Price is trading inside an active bullish imbalance zone", "potential_signal": "Potential long continuation / mitigation entry"},    {"pattern": "Bear FVG Active", "meaning": "Price is trading inside an active bearish imbalance zone", "potential_signal": "Potential short continuation / mitigation entry"},])display(pa_reference)

## 6. Price Action Signal EngineThis block converts raw SNR/SMC features into practical PA-style signal candidates with explanations.

In [ ]:
pa_df = df.copy()pa_df["pa_long_sweep_reversal"] = ((pa_df["bull_sweep"] == 1) & ((pa_df["bull_choch"] == 1) | (pa_df["snr_long_reclaim"] == 1) | (pa_df["in_bull_fvg"] == 1))).astype(int)pa_df["pa_short_sweep_reversal"] = ((pa_df["bear_sweep"] == 1) & ((pa_df["bear_choch"] == 1) | (pa_df["snr_short_reject"] == 1) | (pa_df["in_bear_fvg"] == 1))).astype(int)pa_df["pa_long_breakout"] = ((pa_df["snr_bull_breakout"] == 1) | ((pa_df["bull_break"] == 1) & (pa_df["trend_bull"] == 1))).astype(int)pa_df["pa_short_breakout"] = ((pa_df["snr_bear_breakout"] == 1) | ((pa_df["bear_break"] == 1) & (pa_df["trend_bear"] == 1))).astype(int)pa_df["pa_long_reclaim"] = ((pa_df["snr_long_reclaim"] == 1) | (pa_df["snr_bull_retest"] == 1)).astype(int)pa_df["pa_short_reject"] = ((pa_df["snr_short_reject"] == 1) | (pa_df["snr_bear_retest"] == 1)).astype(int)conditions = [    pa_df["pa_long_sweep_reversal"] == 1,    pa_df["pa_short_sweep_reversal"] == 1,    pa_df["pa_long_breakout"] == 1,    pa_df["pa_short_breakout"] == 1,    pa_df["pa_long_reclaim"] == 1,    pa_df["pa_short_reject"] == 1,]choices = [    "Liquidity Sweep Long",    "Liquidity Sweep Short",    "Bull Breakout",    "Bear Breakout",    "SNR Reclaim Long",    "SNR Reject Short",]pa_df["pa_signal_type"] = np.select(conditions, choices, default="Wait")pa_df["pa_signal_bias"] = np.select([pa_df["pa_signal_type"].str.contains("Long|Bull", regex=True), pa_df["pa_signal_type"].str.contains("Short|Bear", regex=True)], ["Long", "Short"], default="Neutral")pa_df["pa_signal_reason"] = np.select(    conditions,    [        "Liquidity below lows was taken and price reclaimed with bullish confirmation",        "Liquidity above highs was taken and price rejected with bearish confirmation",        "Price is breaking structure / SNR resistance with bullish continuation context",        "Price is breaking structure / SNR support with bearish continuation context",        "Price reclaimed SNR support and is holding above the basis",        "Price rejected SNR resistance and is holding below the basis",    ],    default="No strong PA trigger right now")pa_df["pa_signal_score"] = (    pa_df["bull_sweep"] + pa_df["bear_sweep"] + pa_df["bull_choch"] + pa_df["bear_choch"] +    pa_df["in_bull_fvg"] + pa_df["in_bear_fvg"] + pa_df["snr_long_reclaim"] + pa_df["snr_short_reject"] +    pa_df["snr_bull_breakout"] + pa_df["snr_bear_breakout"] + pa_df["bull_pa_continuation"] + pa_df["bear_pa_continuation"])current_pa_snapshot = pa_df.dropna().iloc[-1]display(pd.Series({    "timestamp": str(current_pa_snapshot.name),    "signal_type": current_pa_snapshot["pa_signal_type"],    "signal_bias": current_pa_snapshot["pa_signal_bias"],    "signal_reason": current_pa_snapshot["pa_signal_reason"],    "signal_score": round(float(current_pa_snapshot["pa_signal_score"]), 2),    "bull_sweep": int(current_pa_snapshot["bull_sweep"]),    "bear_sweep": int(current_pa_snapshot["bear_sweep"]),    "bull_break": int(current_pa_snapshot["bull_break"]),    "bear_break": int(current_pa_snapshot["bear_break"]),    "bull_choch": int(current_pa_snapshot["bull_choch"]),    "bear_choch": int(current_pa_snapshot["bear_choch"]),    "in_bull_fvg": int(current_pa_snapshot["in_bull_fvg"]),    "in_bear_fvg": int(current_pa_snapshot["in_bear_fvg"]),    "snr_long_reclaim": int(current_pa_snapshot["snr_long_reclaim"]),    "snr_short_reject": int(current_pa_snapshot["snr_short_reject"]),})))

In [ ]:
recent_pa_signals = pa_df.loc[pa_df["pa_signal_type"] != "Wait", [    "close", "pa_signal_type", "pa_signal_bias", "pa_signal_reason", "pa_signal_score",    "large_move", "large_up_move", "large_down_move"]].tail(20)display(recent_pa_signals)

## 7. Price Action MapThis chart shows where the market sits relative to SNR, while highlighting PA events like liquidity sweeps, breakouts and CHoCH.

In [ ]:
plot_df = df.dropna(subset=["reg_basis", "snr_upper", "snr_lower", "poc_proxy"]).tail(700).copy()events = plot_df[plot_df["large_move"] == 1]fig, axes = plt.subplots(3, 1, figsize=(18, 14), sharex=True)axes[0].plot(plot_df.index, plot_df["close"], color="#d8dce8", linewidth=1.0, label="Close")axes[0].plot(plot_df.index, plot_df["reg_basis"], color="#b8c2d6", linewidth=1.4, label="SNR Basis")axes[0].plot(plot_df.index, plot_df["snr_upper"], color="#9ca6bd", linewidth=0.9, label="SNR Upper")axes[0].plot(plot_df.index, plot_df["snr_lower"], color="#9ca6bd", linewidth=0.9, label="SNR Lower")axes[0].plot(plot_df.index, plot_df["poc_proxy"], color="#f5b041", linewidth=1.2, label="POC Proxy")axes[0].fill_between(plot_df.index, plot_df["snr_upper"], plot_df["snr_lower"], color="#7a849c", alpha=0.08)axes[0].scatter(events.index, events["close"], s=20, color="#f4b183", label="Future Large Move")axes[0].scatter(plot_df.index[plot_df["bull_sweep"] == 1], plot_df.loc[plot_df["bull_sweep"] == 1, "close"], s=28, color="#2ecc71", label="Bull Sweep")axes[0].scatter(plot_df.index[plot_df["bear_sweep"] == 1], plot_df.loc[plot_df["bear_sweep"] == 1, "close"], s=28, color="#e74c3c", label="Bear Sweep")axes[0].scatter(plot_df.index[plot_df["snr_bull_breakout"] == 1], plot_df.loc[plot_df["snr_bull_breakout"] == 1, "close"], s=36, marker="^", color="#58d68d", label="Bull Breakout")axes[0].scatter(plot_df.index[plot_df["snr_bear_breakout"] == 1], plot_df.loc[plot_df["snr_bear_breakout"] == 1, "close"], s=36, marker="v", color="#ec7063", label="Bear Breakout")axes[0].set_title("BTC Price Action Map: SNR, Sweeps, Breakouts, POC")axes[0].legend(loc="upper left", ncol=6)axes[1].plot(plot_df.index, plot_df["kz_long_score"], color="#3498db", linewidth=1.2, label="KZ Long Score")axes[1].plot(plot_df.index, plot_df["kz_short_score"], color="#9b59b6", linewidth=1.2, label="KZ Short Score")axes[1].plot(plot_df.index, plot_df["flow_main"], color="#f39c12", linewidth=1.2, label="Flow Main")axes[1].set_title("Confluence And Flow For PA Triggers")axes[1].legend(loc="upper left")axes[2].bar(plot_df.index, plot_df["volume"], color="#5dade2", alpha=0.55)axes[2].set_title("Volume Behind Price Action")plt.tight_layout()plt.show()

## 8. Sessions, Kill Zones And Timing ResearchThis block shows where PA-style signals and large moves appear more often by session, hour and day.

In [ ]:
session_stats = pd.DataFrame({    "Asia": [df.loc[df["is_asia"] == 1, "large_move"].mean()],    "London": [df.loc[df["is_london"] == 1, "large_move"].mean()],    "New York": [df.loc[df["is_ny"] == 1, "large_move"].mean()],    "London KZ": [df.loc[df["is_london_kz"] == 1, "large_move"].mean()],    "NY KZ": [df.loc[df["is_ny_kz"] == 1, "large_move"].mean()],    "Silver": [df.loc[df["is_silver"] == 1, "large_move"].mean()],}).T.reset_index()session_stats.columns = ["session", "large_move_rate"]session_stats["large_move_rate"] = session_stats["large_move_rate"].fillna(0.0) * 100hourly_stats = df.groupby("hour")["large_move"].mean().fillna(0.0) * 100dow_stats = df.groupby("day_of_week")["large_move"].mean().fillna(0.0) * 100heatmap = df.groupby(["day_of_week", "hour"])["large_move"].mean().unstack(fill_value=0.0) * 100fig, axes = plt.subplots(2, 2, figsize=(20, 12))sns.barplot(data=session_stats, x="session", y="large_move_rate", palette="Blues_r", ax=axes[0, 0])axes[0, 0].set_title("Large-Move Rate By Session / Kill Zone")axes[0, 0].tick_params(axis="x", rotation=20)axes[0, 1].plot(hourly_stats.index, hourly_stats.values, marker="o", color="#3498db")axes[0, 1].set_title("Large-Move Rate By Hour")axes[0, 1].set_xlabel("Hour")axes[0, 1].set_ylabel("Rate %")axes[1, 0].bar(dow_stats.index.astype(str), dow_stats.values, color="#af7ac5")axes[1, 0].set_title("Large-Move Rate By Day Of Week")sns.heatmap(heatmap, cmap="Blues", ax=axes[1, 1])axes[1, 1].set_title("Large-Move Heatmap By Day And Hour")plt.tight_layout()plt.show()

## 9. Price Action Pattern StatsThis block compares how often PA events like sweeps, breakouts, CHoCH, FVG and SNR reactions lead to large moves.

In [ ]:
research_rows = [    ("Bull BOS", df.loc[df["bull_break"] == 1, "large_move"].mean()),    ("Bear BOS", df.loc[df["bear_break"] == 1, "large_move"].mean()),    ("Bull CHoCH", df.loc[df["bull_choch"] == 1, "large_move"].mean()),    ("Bear CHoCH", df.loc[df["bear_choch"] == 1, "large_move"].mean()),    ("Bull Sweep", df.loc[df["bull_sweep"] == 1, "large_move"].mean()),    ("Bear Sweep", df.loc[df["bear_sweep"] == 1, "large_move"].mean()),    ("Bull FVG Active", df.loc[df["in_bull_fvg"] == 1, "large_move"].mean()),    ("Bear FVG Active", df.loc[df["in_bear_fvg"] == 1, "large_move"].mean()),    ("SNR Long Reclaim", df.loc[df["snr_long_reclaim"] == 1, "large_move"].mean()),    ("SNR Short Reject", df.loc[df["snr_short_reject"] == 1, "large_move"].mean()),    ("Bull PA Continuation", df.loc[df["bull_pa_continuation"] == 1, "large_move"].mean()),    ("Bear PA Continuation", df.loc[df["bear_pa_continuation"] == 1, "large_move"].mean()),]pattern_stats = pd.DataFrame(research_rows, columns=["pattern", "large_move_rate"])pattern_stats["large_move_rate"] = pattern_stats["large_move_rate"].fillna(0.0) * 100pattern_stats = pattern_stats.sort_values("large_move_rate", ascending=False)dist_df = df.copy()dist_df["event_label"] = np.where(dist_df["large_move"] == 1, "Large Move", "No Large Move")dist_df["snr_distance_pct"] = ((dist_df["close"] - dist_df["reg_basis"]) / dist_df["close"]) * 100fig, axes = plt.subplots(2, 2, figsize=(20, 12))sns.barplot(data=pattern_stats, x="large_move_rate", y="pattern", palette="mako", ax=axes[0, 0])axes[0, 0].set_title("Large-Move Rate By Pattern")sns.boxplot(data=dist_df, x="event_label", y="kz_long_score", palette="Blues", ax=axes[0, 1])axes[0, 1].set_title("KZ Long Score vs Event")sns.boxplot(data=dist_df, x="event_label", y="kz_short_score", palette="Purples", ax=axes[1, 0])axes[1, 0].set_title("KZ Short Score vs Event")sns.histplot(data=dist_df, x="snr_distance_pct", hue="event_label", bins=50, stat="density", common_norm=False, ax=axes[1, 1])axes[1, 1].set_title("Distance From SNR Basis (%)")plt.tight_layout()plt.show()display(pattern_stats)

## 10. Potential Signal StatsThis block measures how each potential PA signal performed historically and whether it skewed more to long or short follow-through.

In [ ]:
signal_stats = pa_df.loc[pa_df["pa_signal_type"] != "Wait"].groupby("pa_signal_type").agg(    count=("pa_signal_type", "size"),    large_move_rate_pct=("large_move", lambda s: round(s.mean() * 100, 2)),    up_rate_pct=("large_up_move", lambda s: round(s.mean() * 100, 2)),    down_rate_pct=("large_down_move", lambda s: round(s.mean() * 100, 2)),    avg_signal_score=("pa_signal_score", lambda s: round(s.mean(), 2)),).reset_index().sort_values(["large_move_rate_pct", "count"], ascending=[False, False])display(signal_stats)fig, axes = plt.subplots(1, 2, figsize=(18, 6))sns.barplot(data=signal_stats, x="large_move_rate_pct", y="pa_signal_type", palette="viridis", ax=axes[0])axes[0].set_title("Large Move Rate By Potential PA Signal")sns.barplot(data=signal_stats, x="avg_signal_score", y="pa_signal_type", palette="magma", ax=axes[1])axes[1].set_title("Average Confluence Score By Potential PA Signal")plt.tight_layout()plt.show()

## 11. Price Action PlaybookThis playbook translates the latest market state into a practical PA reading: wait, breakout watch, liquidity sweep long or liquidity sweep short.

In [ ]:
playbook_bias = "WAIT"playbook_reason = "No strong PA alignment yet"playbook_risk = "Stand aside until sweep, breakout or reclaim appears"playbook_target = "Monitor SNR basis, upper/lower channel and recent swing range"if current_pa_snapshot["pa_signal_type"] == "Liquidity Sweep Long":    playbook_bias = "POTENTIAL LONG"    playbook_reason = "Sell-side liquidity was taken and price reclaimed with bullish confirmation"    playbook_risk = "Invalidation below the sweep low / SNR lower support"    playbook_target = "Retest of POC, then SNR upper / range high"elif current_pa_snapshot["pa_signal_type"] == "Liquidity Sweep Short":    playbook_bias = "POTENTIAL SHORT"    playbook_reason = "Buy-side liquidity was taken and price rejected with bearish confirmation"    playbook_risk = "Invalidation above the sweep high / SNR upper resistance"    playbook_target = "Retest of POC, then SNR lower / range low"elif current_pa_snapshot["pa_signal_type"] == "Bull Breakout":    playbook_bias = "BREAKOUT LONG WATCH"    playbook_reason = "Price is expanding above structure / SNR resistance"    playbook_risk = "Watch for failed breakout and fast return below basis"    playbook_target = "Continuation toward fresh highs or next extension leg"elif current_pa_snapshot["pa_signal_type"] == "Bear Breakout":    playbook_bias = "BREAKOUT SHORT WATCH"    playbook_reason = "Price is expanding below structure / SNR support"    playbook_risk = "Watch for failed breakdown and reclaim back above basis"    playbook_target = "Continuation toward fresh lows or next extension leg"elif current_pa_snapshot["pa_signal_type"] == "SNR Reclaim Long":    playbook_bias = "RECLAIM LONG WATCH"    playbook_reason = "Support was defended and price is trying to hold above basis"    playbook_risk = "If basis is lost again, setup weakens quickly"    playbook_target = "POC and upper channel retest"elif current_pa_snapshot["pa_signal_type"] == "SNR Reject Short":    playbook_bias = "REJECT SHORT WATCH"    playbook_reason = "Resistance held and price is trying to stay below basis"    playbook_risk = "If basis is reclaimed, short idea weakens quickly"    playbook_target = "POC and lower channel retest"pa_playbook = pd.Series({    "current_pa_signal": current_pa_snapshot["pa_signal_type"],    "bias": playbook_bias,    "reason": playbook_reason,    "risk": playbook_risk,    "target": playbook_target,    "current_score": round(float(current_pa_snapshot["pa_signal_score"]), 2),})display(pa_playbook)

## 12. Probability Buckets And Target BehaviourThis part looks at target distributions and how future up/down moves are spread across the sample.

In [ ]:
analysis_df = df.dropna(subset=["future_up_move", "future_down_move", "atr14", "close"]).copy()analysis_df["future_max_move"] = analysis_df[["future_up_move", "future_down_move"]].max(axis=1)analysis_df["atr_move_ratio"] = analysis_df["future_max_move"] / analysis_df["atr14"].replace(0, np.nan)analysis_df["vol_bucket"] = pd.qcut(analysis_df["atr14"], q=5, duplicates="drop")bucket_stats = analysis_df.groupby("vol_bucket", observed=False)["large_move"].mean().reset_index()bucket_stats["large_move_rate"] = bucket_stats["large_move"] * 100fig, axes = plt.subplots(2, 2, figsize=(20, 12))sns.histplot(analysis_df["future_up_move"], bins=50, color="#2ecc71", ax=axes[0, 0])axes[0, 0].set_title("Distribution Of Future Up Move")sns.histplot(analysis_df["future_down_move"], bins=50, color="#e74c3c", ax=axes[0, 1])axes[0, 1].set_title("Distribution Of Future Down Move")sns.histplot(analysis_df["atr_move_ratio"].dropna(), bins=50, color="#5dade2", ax=axes[1, 0])axes[1, 0].set_title("Future Max Move / ATR")sns.barplot(data=bucket_stats, x="vol_bucket", y="large_move_rate", palette="crest", ax=axes[1, 1])axes[1, 1].set_title("Large-Move Rate By ATR Bucket")axes[1, 1].tick_params(axis="x", rotation=35)plt.tight_layout()plt.show()

## 13. Live Window DetailThis chart is focused on the most recent market window so it is easier to inspect the current sweep / breakout / reclaim setup without the full history clutter.

In [ ]:
recent_df = df.dropna(subset=["reg_basis", "poc_proxy"]).tail(180).copy()fig, axes = plt.subplots(2, 1, figsize=(18, 10), sharex=True)axes[0].plot(recent_df.index, recent_df["close"], color="#ecf0f1", linewidth=1.2, label="Close")axes[0].plot(recent_df.index, recent_df["reg_basis"], color="#f5b041", linewidth=1.2, label="Basis")axes[0].plot(recent_df.index, recent_df["poc_proxy"], color="#48c9b0", linewidth=1.2, label="POC")axes[0].scatter(recent_df.index[recent_df["bull_sweep"] == 1], recent_df.loc[recent_df["bull_sweep"] == 1, "close"], color="#2ecc71", s=25, label="Bull Sweep")axes[0].scatter(recent_df.index[recent_df["bear_sweep"] == 1], recent_df.loc[recent_df["bear_sweep"] == 1, "close"], color="#e74c3c", s=25, label="Bear Sweep")axes[0].scatter(recent_df.index[recent_df["bull_choch"] == 1], recent_df.loc[recent_df["bull_choch"] == 1, "close"], color="#2ecc71", s=35, label="Bull CHoCH")axes[0].scatter(recent_df.index[recent_df["bear_choch"] == 1], recent_df.loc[recent_df["bear_choch"] == 1, "close"], color="#e74c3c", s=35, label="Bear CHoCH")axes[0].scatter(recent_df.index[recent_df["snr_bull_breakout"] == 1], recent_df.loc[recent_df["snr_bull_breakout"] == 1, "close"], color="#58d68d", s=40, marker="^", label="Bull Breakout")axes[0].scatter(recent_df.index[recent_df["snr_bear_breakout"] == 1], recent_df.loc[recent_df["snr_bear_breakout"] == 1, "close"], color="#ec7063", s=40, marker="v", label="Bear Breakout")axes[0].legend(loc="upper left", ncol=5)axes[0].set_title("Recent Price Action Window")axes[1].plot(recent_df.index, recent_df["kz_long_score"], color="#3498db", label="KZ Long Score")axes[1].plot(recent_df.index, recent_df["kz_short_score"], color="#9b59b6", label="KZ Short Score")axes[1].plot(recent_df.index, recent_df["flow_main"], color="#f39c12", label="Flow Main")axes[1].legend(loc="upper left")axes[1].set_title("Confluence Scores And Flow For Current PA Setup")plt.tight_layout()plt.show()

## 14. Multi-Timeframe Price Action ResearchThis block compares `1m / 5m / 15m` using fresh `yfinance_now` data so you can see where sweeps, breakouts, Silver and SNR reactions behave better.

In [ ]:
MTF_INTERVALS = ["1m", "5m", "15m"]MTF_SOURCE = "yfinance_now"mtf_rows = []mtf_frames = {}for tf in MTF_INTERVALS:    tf_cfg = FeatureConfig(        symbol=SYMBOL,        interval=tf,        period=PERIOD,        timezone=TIMEZONE,        data_source=MTF_SOURCE,        snr_length=200,        outer_band_width=20,        target_horizon=12,        target_threshold=700.0,    )    tf_end_utc = datetime.now(timezone.utc)    tf_df = load_market_data(        csv_path=None,        symbol=tf_cfg.symbol,        interval=tf_cfg.interval,        period=tf_cfg.period,        timezone=tf_cfg.timezone,        data_source=tf_cfg.data_source,        end_datetime=tf_end_utc,        use_yfinance_now=True,    )    tf_df = add_snr_features(tf_df, config=tf_cfg)    tf_df = build_impulse_targets(tf_df, horizon=tf_cfg.target_horizon, move_threshold=tf_cfg.target_threshold)    mtf_frames[tf] = tf_df    tf_train = make_training_frame(tf_df, feature_columns=default_feature_columns(), target_column="large_move")    tf_auc = np.nan    if len(tf_train) > 250 and tf_train["large_move"].nunique() > 1:        tf_split = int(len(tf_train) * 0.8)        tf_X_train = tf_train[default_feature_columns()].iloc[:tf_split]        tf_y_train = tf_train["large_move"].iloc[:tf_split]        tf_X_test = tf_train[default_feature_columns()].iloc[tf_split:]        tf_y_test = tf_train["large_move"].iloc[tf_split:]        if len(tf_X_test) > 0 and tf_y_test.nunique() > 1:            tf_model = HistGradientBoostingClassifier(                learning_rate=0.05,                max_depth=5,                max_iter=180,                min_samples_leaf=30,                random_state=42,            )            tf_model.fit(tf_X_train, tf_y_train)            tf_prob = tf_model.predict_proba(tf_X_test)[:, 1]            tf_auc = roc_auc_score(tf_y_test, tf_prob)    mtf_rows.append({        "timeframe": tf,        "rows": len(tf_df),        "start": str(tf_df.index.min()),        "end": str(tf_df.index.max()),        "large_move_rate_pct": round(tf_df["large_move"].mean() * 100, 2),        "ny_rate_pct": round(tf_df.loc[tf_df["is_ny"] == 1, "large_move"].mean() * 100, 2),        "silver_rate_pct": round(tf_df.loc[tf_df["is_silver"] == 1, "large_move"].mean() * 100, 2),        "bull_fvg_rate_pct": round(tf_df.loc[tf_df["in_bull_fvg"] == 1, "large_move"].mean() * 100, 2),        "bear_fvg_rate_pct": round(tf_df.loc[tf_df["in_bear_fvg"] == 1, "large_move"].mean() * 100, 2),        "snr_long_rate_pct": round(tf_df.loc[tf_df["snr_long_reclaim"] == 1, "large_move"].mean() * 100, 2),        "snr_short_rate_pct": round(tf_df.loc[tf_df["snr_short_reject"] == 1, "large_move"].mean() * 100, 2),        "event_auc": round(float(tf_auc), 4) if pd.notna(tf_auc) else np.nan,        "bull_sweep_rate_pct": round(tf_df.loc[tf_df["bull_sweep"] == 1, "large_move"].mean() * 100, 2),        "bear_sweep_rate_pct": round(tf_df.loc[tf_df["bear_sweep"] == 1, "large_move"].mean() * 100, 2),        "bull_breakout_rate_pct": round(tf_df.loc[tf_df["snr_bull_breakout"] == 1, "large_move"].mean() * 100, 2),        "bear_breakout_rate_pct": round(tf_df.loc[tf_df["snr_bear_breakout"] == 1, "large_move"].mean() * 100, 2),    })mtf_table = pd.DataFrame(mtf_rows)display(mtf_table)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 11))sns.barplot(data=mtf_table, x="timeframe", y="large_move_rate_pct", palette="Blues_r", ax=axes[0, 0])axes[0, 0].set_title("Large-Move Rate By Timeframe")sns.barplot(data=mtf_table, x="timeframe", y="silver_rate_pct", palette="Purples", ax=axes[0, 1])axes[0, 1].set_title("Silver Large-Move Rate By Timeframe")sns.barplot(data=mtf_table, x="timeframe", y="snr_long_rate_pct", palette="Greens", ax=axes[1, 0])axes[1, 0].set_title("SNR Long Reclaim Rate By Timeframe")sns.barplot(data=mtf_table, x="timeframe", y="event_auc", palette="mako", ax=axes[1, 1])axes[1, 1].set_title("Event ROC AUC By Timeframe")plt.tight_layout()plt.show()fig, axes = plt.subplots(len(MTF_INTERVALS), 1, figsize=(18, 4 * len(MTF_INTERVALS)), sharex=False)axes = np.atleast_1d(axes)for ax, tf in zip(axes, MTF_INTERVALS):    tf_recent = mtf_frames[tf].dropna(subset=["reg_basis", "snr_upper", "snr_lower"]).tail(180)    ax.plot(tf_recent.index, tf_recent["close"], color="#d8dce8", linewidth=1.0, label=f"{tf} Close")    ax.plot(tf_recent.index, tf_recent["reg_basis"], color="#f5b041", linewidth=1.0, label="Basis")    ax.plot(tf_recent.index, tf_recent["poc_proxy"], color="#48c9b0", linewidth=1.0, label="POC")    ax.fill_between(tf_recent.index, tf_recent["snr_upper"], tf_recent["snr_lower"], color="#7a849c", alpha=0.08)    ax.set_title(f"Recent SNR Context: {tf}")    ax.legend(loc="upper left", ncol=4)plt.tight_layout()plt.show()

## 15. Best Setup FinderThis block ranks the strongest setup combinations by timeframe using session, Silver, FVG, SNR and structure / PA signals.

In [ ]:
setup_rules = [    ("NY + Silver", lambda d: (d["is_ny"] == 1) & (d["is_silver"] == 1)),    ("London KZ + Bull FVG", lambda d: (d["is_london_kz"] == 1) & (d["in_bull_fvg"] == 1)),    ("NY KZ + Bear FVG", lambda d: (d["is_ny_kz"] == 1) & (d["in_bear_fvg"] == 1)),    ("Silver + SNR Long", lambda d: (d["is_silver"] == 1) & (d["snr_long_reclaim"] == 1)),    ("Silver + SNR Short", lambda d: (d["is_silver"] == 1) & (d["snr_short_reject"] == 1)),    ("Bull CHoCH + Bull FVG", lambda d: (d["bull_choch"] == 1) & (d["in_bull_fvg"] == 1)),    ("Bear CHoCH + Bear FVG", lambda d: (d["bear_choch"] == 1) & (d["in_bear_fvg"] == 1)),    ("Bull Sweep + SNR Long", lambda d: (d["bull_sweep"] == 1) & (d["snr_long_reclaim"] == 1)),    ("Bear Sweep + SNR Short", lambda d: (d["bear_sweep"] == 1) & (d["snr_short_reject"] == 1)),    ("Bull PA + Discount", lambda d: (d["bull_pa_continuation"] == 1) & (d["in_discount"] == 1)),    ("Bear PA + Premium", lambda d: (d["bear_pa_continuation"] == 1) & (d["in_premium"] == 1)),]best_setup_rows = []for tf, tf_df in mtf_frames.items():    for rule_name, rule_fn in setup_rules:        mask = rule_fn(tf_df).fillna(False)        sample = tf_df.loc[mask].copy()        count = len(sample)        if count < 8:            continue        best_setup_rows.append({            "timeframe": tf,            "setup": rule_name,            "count": count,            "large_move_rate_pct": round(sample["large_move"].mean() * 100, 2),            "up_rate_pct": round(sample["large_up_move"].mean() * 100, 2),            "down_rate_pct": round(sample["large_down_move"].mean() * 100, 2),            "avg_kz_long_score": round(sample["kz_long_score"].mean(), 2),            "avg_kz_short_score": round(sample["kz_short_score"].mean(), 2),        })best_setup_table = pd.DataFrame(best_setup_rows)best_setup_table = best_setup_table.sort_values(["timeframe", "large_move_rate_pct", "count"], ascending=[True, False, False])best_setup_top = best_setup_table.groupby("timeframe", group_keys=False).head(5).reset_index(drop=True)display(best_setup_top)

In [ ]:
fig, axes = plt.subplots(1, len(MTF_INTERVALS), figsize=(6 * len(MTF_INTERVALS), 6), sharex=False)axes = np.atleast_1d(axes)for ax, tf in zip(axes, MTF_INTERVALS):    tf_best = best_setup_top.loc[best_setup_top["timeframe"] == tf].copy()    if tf_best.empty:        ax.set_title(f"Best Setups: {tf}")        ax.text(0.5, 0.5, "No qualifying setups", ha="center", va="center")        ax.axis("off")        continue    sns.barplot(data=tf_best, x="large_move_rate_pct", y="setup", palette="viridis", ax=ax)    ax.set_title(f"Best Setups: {tf}")    ax.set_xlabel("Large Move Rate %")plt.tight_layout()plt.show()

## 16. Secondary AI CheckThe event model is kept here only as a secondary confirmation layer after the Price Action reading.

In [ ]:
feature_cols = default_feature_columns()train_df = make_training_frame(df, feature_columns=feature_cols, target_column="large_move")split_idx = int(len(train_df) * 0.8)X_train = train_df[feature_cols].iloc[:split_idx]y_train = train_df["large_move"].iloc[:split_idx]X_test = train_df[feature_cols].iloc[split_idx:]y_test = train_df["large_move"].iloc[split_idx:]event_model = HistGradientBoostingClassifier(    learning_rate=0.05,    max_depth=6,    max_iter=250,    min_samples_leaf=40,    random_state=42,)event_model.fit(X_train, y_train)event_prob = event_model.predict_proba(X_test)[:, 1]event_pred = (event_prob >= 0.50).astype(int)print("ROC AUC:", round(roc_auc_score(y_test, event_prob), 4))print("PR AUC:", round(average_precision_score(y_test, event_prob), 4))print(classification_report(y_test, event_pred, digits=4))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 12))RocCurveDisplay.from_predictions(y_test, event_prob, ax=axes[0, 0])axes[0, 0].set_title("Event ROC Curve")PrecisionRecallDisplay.from_predictions(y_test, event_prob, ax=axes[0, 1])axes[0, 1].set_title("Event Precision-Recall")ConfusionMatrixDisplay.from_predictions(y_test, event_pred, ax=axes[1, 0], cmap="Blues")axes[1, 0].set_title("Event Confusion Matrix")perm = permutation_importance(event_model, X_test, y_test, n_repeats=5, random_state=42, scoring="roc_auc")importance_df = pd.DataFrame({"feature": feature_cols, "importance": perm.importances_mean}).sort_values("importance", ascending=False).head(20)sns.barplot(data=importance_df, x="importance", y="feature", palette="Blues_r", ax=axes[1, 1])axes[1, 1].set_title("Top 20 Event Features")plt.tight_layout()plt.show()display(importance_df)

## 17. Secondary Direction AI CheckThe direction model is a secondary check for whether the follow-through after a PA trigger is more likely up or down.

In [ ]:
direction_df = train_df.join(df[["large_up_move", "large_down_move"]], how="left")direction_df = direction_df.loc[(direction_df["large_up_move"] == 1) | (direction_df["large_down_move"] == 1)].copy()direction_df["up_direction"] = direction_df["large_up_move"].astype(int)dir_split = int(len(direction_df) * 0.8)X_dir_train = direction_df[feature_cols].iloc[:dir_split]y_dir_train = direction_df["up_direction"].iloc[:dir_split]X_dir_test = direction_df[feature_cols].iloc[dir_split:]y_dir_test = direction_df["up_direction"].iloc[dir_split:]direction_model = HistGradientBoostingClassifier(    learning_rate=0.05,    max_depth=5,    max_iter=200,    min_samples_leaf=30,    random_state=42,)direction_model.fit(X_dir_train, y_dir_train)dir_prob = direction_model.predict_proba(X_dir_test)[:, 1]dir_pred = (dir_prob >= 0.50).astype(int)print("Direction ROC AUC:", round(roc_auc_score(y_dir_test, dir_prob), 4))print(classification_report(y_dir_test, dir_pred, digits=4))fig, axes = plt.subplots(1, 3, figsize=(18, 5))RocCurveDisplay.from_predictions(y_dir_test, dir_prob, ax=axes[0])axes[0].set_title("Direction ROC Curve")PrecisionRecallDisplay.from_predictions(y_dir_test, dir_prob, ax=axes[1])axes[1].set_title("Direction Precision-Recall")ConfusionMatrixDisplay.from_predictions(y_dir_test, dir_pred, ax=axes[2], cmap="Purples")axes[2].set_title("Direction Confusion Matrix")plt.tight_layout()plt.show()

## 18. Latest AI Export And TradingView BridgeThis section produces the latest model output and writes the same bridge files used by the indicator workflow.

In [ ]:
latest_features = df[feature_cols].dropna().iloc[[-1]]latest_row = df.loc[latest_features.index[0]]latest_event_prob = float(event_model.predict_proba(latest_features)[0, 1])latest_up_prob = float(direction_model.predict_proba(latest_features)[0, 1])latest_down_prob = 1.0 - latest_up_probpayload = build_signal_payload(    latest_row,    large_move_probability=latest_event_prob,    up_move_probability=latest_up_prob,    down_move_probability=latest_down_prob,    symbol=SYMBOL,    timeframe=INTERVAL,    notes="Notebook export aligned with SNR_Line.pine",)indicator_fields = payload_to_indicator_fields(payload)bridge_payload = build_pine_bridge_payload(payload)(OUT_DIR / "latest_signal.json").write_text(json.dumps(payload.__dict__, indent=2), encoding="utf-8")(OUT_DIR / "indicator_snapshot.json").write_text(json.dumps(indicator_fields, indent=2), encoding="utf-8")(OUT_DIR / "tradingview_ai_bridge.txt").write_text(bridge_payload, encoding="utf-8")display(pd.Series({**indicator_fields, "pine_bridge_payload": bridge_payload}))

## 19. Live Data MonitorUse this if you want periodic refreshes of the current frame without re-running the whole notebook manually.

In [ ]:
def latest_live_snapshot(live_df):    last = live_df.dropna().iloc[-1]    return pd.Series({        "timestamp": str(last.name),        "close": round(float(last["close"]), 2),        "reg_basis": round(float(last["reg_basis"]), 2),        "snr_upper": round(float(last["snr_upper"]), 2),        "snr_lower": round(float(last["snr_lower"]), 2),        "poc_proxy": round(float(last["poc_proxy"]), 2),        "bull_choch": int(last["bull_choch"]),        "bear_choch": int(last["bear_choch"]),        "in_bull_fvg": int(last["in_bull_fvg"]),        "in_bear_fvg": int(last["in_bear_fvg"]),        "kz_long_score": round(float(last["kz_long_score"]), 2),        "kz_short_score": round(float(last["kz_short_score"]), 2),    })def monitor_live_data(iterations=10, sleep_seconds=LIVE_REFRESH_SECONDS):    for step in range(iterations):        live_df, live_utc = load_research_frame()        clear_output(wait=True)        print(f"Live refresh {step + 1}/{iterations} | pulled_at_utc={live_utc.isoformat()} | source={cfg.data_source} | interval={cfg.interval}")        display(latest_live_snapshot(live_df))        time.sleep(sleep_seconds)display(latest_live_snapshot(df))

## 20. Binance Websocket ModeThis mode is optional and is useful when you want a near real-time Binance kline stream inside the notebook.

In [ ]:
WS_SYMBOL = "BTCUSDT"WS_INTERVAL = INTERVAL# Run this cell only when you want streaming:# ws_stream = BinanceKlineStream(symbol=WS_SYMBOL, interval=WS_INTERVAL, max_events=500).start()# ws_stream.wait_for_messages(min_messages=2, timeout=10)# display(pd.Series(ws_stream.latest_snapshot()))

In [ ]:
def monitor_websocket_stream(stream, iterations=20, sleep_seconds=2):    for step in range(iterations):        clear_output(wait=True)        snapshot = pd.Series(stream.latest_snapshot())        print(f"Websocket refresh {step + 1}/{iterations} | symbol={WS_SYMBOL} | interval={WS_INTERVAL}")        display(snapshot)        recent = stream.recent_frame(limit=5)        if not recent.empty:            display(recent[["event_time", "open_time", "close", "volume", "is_closed"]].tail())        time.sleep(sleep_seconds)# When finished: ws_stream.stop()

## 21. Screenshot Fallback ResearchIf fresh structured data is unavailable, the project can still parse a TradingView screenshot from `inbox/`.

In [ ]:
inbox_dir = DATA_DIR / "inbox"latest_shot = choose_latest_screenshot(inbox_dir)if latest_shot is None:    print("No screenshots found in inbox/")else:    shot = analyze_screenshot(latest_shot)    display(pd.Series(shot.__dict__))

## 22. Reading The NotebookSuggested order:1. Run config and compare `auto` vs `yfinance_now`2. Read the current market snapshot and PA reference3. Check current `Price Action Signal Engine` and `Playbook`4. Study sweeps, breakouts, CHoCH, MTF and best setup statistics5. Use AI sections only as a secondary confirmation layer6. Export the latest bridge payload for TradingView